# Attention and Transformers

In previous weeks, we introduced the LSTM as an important practical implementation of recurrent neural networks. LSTMs are effective at handling sequential data—particularly when there are long-range dependencies between elements in a data stream. For several years, one important application of LSTM-based models was in so-called encoder–decoder architectures for tasks such as machine translation.

However, LSTM models are no longer state-of-the-art in sequential data processing. Instead, Transformer-style architectures—particularly models such as BERT—now dominate the processing of text, as well as other forms of sequential data.

A key ingredient of the Transformer approach is the concept of attention. Today, we will introduce the notion of attention and explore the key ideas underlying the Transformer architecture.

We begin with one final look at LSTMs (and image processing) to help build an intuitive understanding of what attention is and why it is useful.

Attention in Sequence-to-Sequence Models

In previous weeks, we briefly introduced the sequence-to-sequence architecture as a type of encoder–decoder architecture. In a sequence-to-sequence architecture, the input is a sequence (e.g., a text string), and the output is another sequence (e.g., a translated text string). This type of architecture has a wide range of applications, including machine translation, question answering, and chatbots.

Sequence-to-Sequence Illustration

To illustrate the sequence-to-sequence framework, let us assume our task is machine translation, where the source sentence $S1$ and the target sentence $S2$ come from different languages. The basic approach to an RNN-centric sequence-to-sequence architecture is illustrated below:

<!-- seq2seq -->
<img width="500" src="https://drive.google.com/uc?id=1x8vGitl11oyXrhq2EKGYfc1Fu5n95zyG"/>

On the left-hand side, we have an LSTM implementation (shown unfolded over time). The input, or encoder LSTM, processes the text one symbol at a time. This produces a hidden state that captures the meaning of the input sequence. Normally, an LSTM would also generate output symbols; however, in this case, we are not training the encoder LSTM against a specific output. Therefore, we do not explicitly show outputs, although they are still produced internally.

Rather than focusing on these outputs, we are primarily interested in the hidden state built up by the encoder—this is the key element that is passed to the decoder.

Turning to the decoder on the right-hand side of the image, we again have an LSTM—distinct from the encoder. The decoder LSTM is initialised with the hidden state produced by the encoder. It then begins generating output symbols, similar to what we saw in the language generation example. Unlike the encoder, the decoder does not have a separate input sequence; instead, its own outputs are fed back into it one symbol at a time.

From this, we can see that the architecture consists of two LSTMs sharing a hidden state. This structure is commonly referred to as an encoder–decoder architecture, as the task is clearly divided between:

- an encoder that builds a representation of the input, and
- a decoder that maps this representation to the target output.

Beyond machine translation, two other important applications of sequence-to-sequence encoder–decoder models are chatbot systems and data-to-text generation.

### Challenges with RNN based Encoder-Decoders

While LSTMs are significantly more powerful than simple RNNs, they still suffer from the same fundamental issue—namely, handling long-range dependencies in data.

To put it another way, imagine being a translator tasked with converting a full sentence into a target language. If the sentence is five or ten words long, this is relatively straightforward. However, as sentences become longer, the challenge of keeping all the relevant information “in your head” and producing an accurate translation becomes much more difficult. In neural network terms, this means that information from words at the beginning of a long sentence must propagate through many layers before influencing words at the end of the target sentence. This creates difficulties during training, as exploding and vanishing gradients remain a challenge in very deep or long sequence models.

The practical impact of this issue can be observed in machine translation performance across sentences of varying lengths. For example, a graph from a well-known paper plots input sentence length against BLEU (pronounced “blue”) score, a common evaluation metric for translation quality. (At this stage, you only need to know that higher scores are better.) The results show that the system performs well on medium-length sentences, but struggles with both very short and longer sentences.

<!-- bleu.png -->
<img width="500" src="https://drive.google.com/uc?id=1E9896BUYRDOy82WPdQRZb8X7XZmdM41I"/>

This is a relatively fundamental problem for LSTMs. It is not limited to language translation; it arises whenever we use an RNN-based sequence-to-sequence architecture with long input sequences.

### Using Attention in Sequence to Sequence Architectures

The fundamental problem with sequence-to-sequence architectures is that the entire input must be compressed into a single hidden representation, which is then passed to the decoder to generate the output sentence. This hidden state corresponds to the symbol $S$ in the diagram above. Ideally, this state variable would capture all the necessary information; however, in practice, it often fails to do so.

What we would prefer is that, when generating each output word, the decoder has access to the specific parts of the input that are most relevant at that step. For example, when producing the first word of the output, it would be beneficial for the decoder to focus on the parts of the input most strongly related to that word. In our example, the output word “Er” would be highly influenced by the input word “He”, as illustrated below.


<img width="500" src="https://drive.google.com/uc?id=10G88K2ONcENzsZBymePr0p2tY7bNG8uN"/>

In this setup, the decoder does not rely solely on the final encoded state. Instead, it is allowed to peek back at the encoder states corresponding to different points in the input sequence.

For this to work, the decoder must have access to the encoder’s hidden states at every time step. At first, this might seem problematic, since sequence models process inputs step by step and do not explicitly “look back” in time. However, in practice, we unroll the RNN up to a specified maximum sequence length during training, which means that all intermediate states are available. Therefore, the decoder can access the full set of encoder states as needed.

This more general setup is illustrated in the figure below:

<img width="500" src="https://drive.google.com/uc?id=10MCyh-1niwGWsxpIt3InV_EHf1ju7yGK"/>

This setup is, in some ways, similar to a fully feedforward network, except that we consume the input one element of the sequence at a time. This is beneficial because we effectively have access to the entire set of information. If we use a suitable representation—such as word embeddings—to encode each input word, we obtain a rich and meaningful representation of the sequence.

However, allowing a fully feedforward-style approach would provide too much information all at once, which is not ideal. For tasks such as translation, we typically only need to focus on specific parts of the input at any given time. In other words, when generating a particular output token, we do not want to attend equally to the entire input sequence; instead, we want to focus on the most relevant parts.

This is where the concept of attention becomes important. With an attention mechanism, we retain access to a rich representation of the entire input, but we introduce a way to selectively focus on the most relevant components. Importantly, attention does not replace the standard network weights; rather, it provides an additional mechanism that operates alongside them.

The baseline attention mechanism is illustrated in the figure below.

<img width="500" src="https://drive.google.com/uc?id=10MI4c3lQVCI_aFNcX3UsddzMCqmdSL_w"/>


The implementation of attention is, in practice, complex and beyond the scope of this discussion. However, the intuition behind it is both important and highly relevant. Rather than treating all input states equally—for example, by summing or averaging them—we instead assign different weights to different parts of the input sequence. These weights vary depending on the specific step in the output generation process, allowing the model to focus more on the most relevant inputs at each stage.


## Attention Models in Image Processing

The use of attention and Transformers is not limited to text problems, nor to strictly encoder–decoder or sequence-to-sequence tasks.

In fact, it can sometimes be easier to understand attention through its application in image captioning. In image captioning, the input is an image that is processed through a traditional CNN pipeline. As you may recall, this produces a set of feature maps for the image, where each feature map represents the presence of a particular feature across different regions of the image.

We could take these feature maps and feed them into a language decoder. In theory, this would provide the key architectural elements for generating captions. However, the challenge is that, without attention, the decoder would have to consider the entire feature map every time it generates a word, which can be inefficient and may dilute focus on the most relevant parts of the image.


<img width="500" src="https://drive.google.com/uc?id=10PsBD4ep3sU9nZNXd2SdThOxn46ES-__"/>

By adding an attention mechanism, we allow the network to learn to focus on specific parts of the image as the decoder generates each word in the caption. This approach was demonstrated in the seminal “Show, Attend and Tell” paper, where the authors trained the network to selectively attend to different regions of the input image at each step of the language generation process.


<img width="500" src="https://drive.google.com/uc?id=10Ri5AjBykLt6G8N5O9dkRFdl44N7qS1i"/>

We can visualise this by having a look at a number of images with generated captions which have been augmented to show where the attention layer is focusing on most on the original input image.


<img width="600" src="https://drive.google.com/uc?id=1t-dQbIYeFJXd9vVegDyVBxyslWo-PA8k"/>

Note that the full set of features is generated by the image backbone network (CNN), and we have access to all of these features across the entire input image. The attention layer does not replace these features; rather, it controls how much focus we place on different parts of the image at each step.

In this way, the attention mechanism is both global and local simultaneously:

- Global: it considers the full input image at all times.
- Local: it allows the model to focus selectively on the most relevant regions of the image for generating each word.



## Transformers

Attention, as we have seen applied to LSTM inputs and images, fundamentally changed deep learning by allowing models to consume entire inputs while selectively focusing on relevant parts. This was far more flexible than traditional RNN-based models. However, the basic attention mechanism was only the beginning.

Researchers soon realized that LSTMs were not strictly necessary. In fact, attention could be applied to enhanced feedforward networks without the sequential RNN component. The intuition is that the sequential processing of the encoder–decoder architecture was not essential; instead, the model could rely primarily on attention, with some additional mechanisms to preserve the ordering of the input.

This idea was formalized in the paper “Attention Is All You Need”, which introduced the original Transformer architecture. The Transformer architecture is illustrated below, and we will discuss it in detail shortly. It is important to note that while this is the original Transformer, many variants and extensions have since been developed. These are all commonly referred to as Transformer architectures, but the model below was the originator.


<img width="300" src="https://drive.google.com/uc?id=10Njcc9HUUBXuEp7UugItRHgvp_g8Yi9q"/>

The Transformer architecture is an encoder–decoder model designed to transform one sequence of text into another, and it was originally developed for machine translation tasks. Unlike the RNN/LSTM architectures we saw earlier, the Transformer processes the entire input sequence in a single step, from which it can generate a high-quality sentence embedding.

The advantage of processing the input in a single step is that the decoder always has a short path to any part of the input, whether that information comes from the start or the end of the input sequence. This overcomes the long-range dependency issues that LSTMs face.

In the diagram, the encoder is on the left-hand side, while the decoder is on the right-hand side. This distinction between encoder and decoder is illustrated in the figure below, where we have included an example input text.


<img width="600" src="https://drive.google.com/uc?id=10P1qN_JzlRMXWMIxFsEbio0SO4S1jl-H"/>

The input text is initially fed into the network in a traditional way—typically with each word transformed into a distributed word embedding via a pre-trained embedding layer. These word embeddings are then combined with positional embeddings, which encode the position of each word in the sequence. In other words, the embedding for a word captures not only its semantic meaning but also its location in the sentence. This combined vector is then processed through subsequent layers to build a complete sentence representation.

A major innovation of the Transformer's encoder is the use of multi-headed attention layers, which allow the network to learn relationships between words within the input. While the detailed mechanics of multi-headed attention are beyond our scope here, it can be thought of as an associative memory mechanism that enables the model to learn which words should focus on others during encoding.

At the end of the encoding process, a representation of the entire input sequence is passed to the decoder. Like the encoder, the decoder receives a representation of the full input sentence. However, unlike the encoder, the decoder operates sequentially: it generates one word at a time.

At each step, the decoder outputs a word via a softmax layer.
The generated word is then fed back into the decoder as additional input for the next step.

Thus, at any given time, the decoder has access to:

- A distributed sentence embedding of the entire input.
- A representation of the decoded output so far (up to the nth word).

Using this information, the decoder predicts the n+1th word, which is then added to the decoded sequence for the next step.


### Beyond the baseline Transformer

The basic Transformer architecture can be used as a single model, but its encoder and decoder components have led to a variety of model variants.

- BERT and similar models can be thought of as implementations of the Transformer’s encoder. These models stack multiple instances of the multi-headed attention + feed-forward network blocks. The encoder’s role is to build a high-quality representation of the input sentence, which is why pre-trained language representations from Google, HuggingFace, and others are essentially robust Transformer encoders.
- The decoder, on the other hand, forms the basis of a large class of generative models, such as GPT-3. These models can generate text and answer a wide variety of queries, having been trained primarily on large-scale text data from web crawls. Generative language models, including ChatGPT, are therefore practical applications of the Transformer decoder architecture.

In practice, models like ChatGPT include more sophisticated training mechanisms, such as reinforcement learning and human-in-the-loop feedback for ranking outputs. Nevertheless, the underlying Transformer decoder architecture remains the foundation.

### Training the basic Transformer

The training of the transformer architecture is via a number of language processing tasks, most significantly a blanked out word task is used so that the network can learn to predict missing words from context.

## Further Reading

For those interested in reading more, here are some good resources to get you started:

Two excellent blog posts on attention:
https://blog.floydhub.com/attention-mechanism/
https://lilianweng.github.io/lil-log/2018/06/24/attention-attention.html

A very good overview of how the BERT Transformer works
https://www.youtube.com/watch?v=4Bdc55j80l8

Simple introduction to Transformers:
https://www.youtube.com/watch?v=FWFA4DGuzSc
